In [22]:
import pandas as pd

df = pd.read_excel('./data/Skylines_GameData.xlsx', sheet_name="Skylines_GameData")


In [23]:
df.head()

,anon_id,AGE,GENDER,LOCATION,EXPERTISE,GAME_REVISION,CITY,SESSION_ID,game,YEAR,...,OPTION_3,OPTION_4,env_before_policy,health_before_policy,satisf_before_policy,ecny_before_policy,env_after_policy,health_after_policy,satisf_after_policy,ecny_after_policy
0,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,1,...,NaN,NaN,12.0,36.0,26.0,19.0,14.0,34.0,27.0,19.0
1,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,2,...,NaN,NaN,14.0,34.0,27.0,19.0,10.0,30.0,28.0,18.0
2,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,3,...,NaN,NaN,10.0,30.0,28.0,18.0,14.0,32.0,29.0,18.0
3,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,4,...,NaN,NaN,14.0,32.0,29.0,18.0,22.0,39.0,22.0,12.0
4,u_1,50-64,Male,Amsterdam,4,4003,Amsterdam,wPXclC4pWk,6445.0,5,...,Encourage use of heat pumps to lower fuel bills,Subsidise people to replace old cars with hybrids,22.0,39.0,22.0,12.0,22.0,39.0,22.0,12.0


In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173516 entries, 0 to 173515
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   anon_id               173516 non-null  object 
 1   AGE                   173516 non-null  object 
 2   GENDER                173516 non-null  object 
 3   LOCATION              173516 non-null  object 
 4   EXPERTISE             173516 non-null  object 
 5   GAME_REVISION         173516 non-null  int64  
 6   CITY                  173516 non-null  object 
 7   SESSION_ID            173125 non-null  object 
 8   game                  135920 non-null  float64
 9   YEAR                  173516 non-null  int64  
 10  AREA                  173516 non-null  object 
 11  TYPE                  173516 non-null  object 
 12  CHOSEN_ID             173516 non-null  int64  
 13  OPTION_1_ID           173516 non-null  int64  
 14  OPTION_2_ID           173516 non-null  int64  
 15  

Lots of missing values, that we will treat right now. 
We won't drop any columns, as all of them can be source of further discussion.

In [25]:
BEFORE_POLICY = "_before_policy"
AFTER_POLICY = "_after_policy"

In [26]:
def calc_completed(serie, suffix) :
    return sum([1 for elem in [serie["env"+suffix], 
            serie["health"+suffix], 
            serie["satisf"+suffix], 
            serie["ecny"+suffix]] if not pd.isna(elem)])    
# spent way too much time realizing that imported data is stored as np.float first
# and that checking "is None" never hit

In [27]:
completeness_rows = []

for anon_id, group in df.groupby("anon_id", sort=False):
    before_completed = sum(
        calc_completed(row, BEFORE_POLICY)
        for _, row in group.iterrows()
    )
    after_completed = sum(
        calc_completed(row, AFTER_POLICY)
        for _, row in group.iterrows()
    )

    before_total = len(group) * 4
    after_total = len(group) * 4

    completeness_rows.append({
        "anon_id": anon_id,
        "before_completed": before_completed,
        "before_total": before_total,
        "after_completed": after_completed,
        "after_total": after_total,
    })

completeness = pd.DataFrame(completeness_rows)

completeness["before_percent"] = completeness["before_completed"] / completeness["before_total"]
completeness["after_percent"] = completeness["after_completed"] / completeness["after_total"]
completeness["ba_percent"] = (completeness["before_completed"] + completeness["after_completed"]) / (completeness["before_total"] + completeness["after_total"]) 

In [28]:
total = 0
for index, row in completeness.iterrows() :
    if row["after_percent"] <= 0.8 : 
        total += 1
total

865

In [29]:
total = 0
for index, row in completeness.iterrows() : 
    if row["before_percent"] <= 0.8 :
        total += 1

total

865

We end up with around 865 users that are 80% full, so we might as well drop them. We can try to look up for a correlation between missing values and other variables but it would take a long time.

In [35]:
SCORE_COLS = [c for c in df.columns if c.endswith(("_before_policy", "_after_policy"))]
m = df[SCORE_COLS].isna()

In [36]:
m.sum()

env_before_policy       37596
health_before_policy    37596
satisf_before_policy    37596
ecny_before_policy      37596
env_after_policy        37596
health_after_policy     37596
satisf_after_policy     37596
ecny_after_policy       37596
dtype: int64

I might just drop all of them, as they are not of any use for us, as missing values can be tough to reproduce. We would need to do some machine learning to understand the variations of each options, and then replicate them to fill the gaps, which, as you can notice, is quite a pain to deal with.

In [39]:
df_clean = df.dropna(axis=0, subset=SCORE_COLS, how="any")
# "how" means if there is a NA, drop the line/col
# axis = 0 -> line / axis = 1 -> col

In [ ]:
m = df_clean[SCORE_COLS].isna()
m.sum()

env_before_policy       0
health_before_policy    0
satisf_before_policy    0
ecny_before_policy      0
env_after_policy        0
health_after_policy     0
satisf_after_policy     0
ecny_after_policy       0
dtype: int64

We come to the end of this cleaning. It was a pain to deal with this data, but it is now safe to say that we should drop it.

Duplicates are crucial and should be kept, as they highlight similar profiles between players, which is part of our objective.